# Topic 3 - SemCor and cross-language frequency propagation

The topic with the real analysis: two distinct frequency signals (concept
commonness vs lemma token frequency) and whether the English sense-tagged
counts propagate across languages. See [`05.54_data_enrich.md`](../../scratch_space/09_concept_model/05.54_data_enrich/05.54_data_enrich.md) Topic 3.

Open questions: SemCor coverage and skew; concept-commonness plausibility;
does it correlate with an independent per-language frequency list; does the
English signal transfer to es / it lemma frequency.

## Setup

Thin caller over the staged cache and the OMW wordnets.

In [ ]:
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import wn
from loguru import logger as lg

from lang_tools.lexicon.ingestion.sources.omw import OMW_LEXICONS, OMW_VERSION
from lang_tools.params.lang_tools_params import get_lang_tools_params

LANGS = ["en", "pt", "es", "fr", "it"]
paths = get_lang_tools_params().paths
data_fol = paths.data_fol
staging = data_fol / "_raw/lexicon/staging"
wn.config.data_directory = str(data_fol / "_raw/lexicon/wn_data")


def staged(dataset: str, lang: str) -> pd.DataFrame:
    """Read a staged parquet (``<staging>/<dataset>/<lang>.parquet``)."""
    return pq.read_table(staging / dataset / f"{lang}.parquet").to_pandas()


def wordnet(lang: str) -> wn.Wordnet:
    """Open the pinned OMW lexicon for a language (one wordnet, no merging)."""
    return wn.Wordnet(lexicon=f"{OMW_LEXICONS[lang]}:{OMW_VERSION}")


def ili_of(synset: wn.Synset) -> str | None:
    """Return the synset's ILI id as a plain string, or ``None``."""
    il = synset.ili
    return getattr(il, "id", il) or None


lg.info("staging at {}", staging)

## SemCor coverage + skew

In [ ]:
# SemCor sense-count coverage + skew (English only; sense.counts()).
counts = []
for se in wordnet("en").senses():
    c = se.counts()
    if c:
        counts.append(sum(c))
counts = np.array(counts)
n_senses = sum(1 for _ in wordnet("en").senses())
print("en senses:", n_senses, "with count:", len(counts),
      f"({100 * len(counts) / n_senses:.1f}%)")
print("count total:", int(counts.sum()),
      "median:", int(np.median(counts)),
      "p90:", int(np.percentile(counts, 90)),
      "max:", int(counts.max()))
# non-en lexicons carry no counts (expected): spot-check it.
print("it senses with a count:",
      sum(1 for se in wordnet("it").senses() if se.counts()))

## Concept commonness vs the en frequency list

In [ ]:
# Concept commonness: sum SemCor counts over the senses of each ILI (en),
# then correlate against the independent en frequency list (max lemma zipf).
ili_count: dict[str, int] = defaultdict(int)
ili_lemmas: dict[str, set[str]] = defaultdict(set)
for s in wordnet("en").synsets():
    il = ili_of(s)
    if not il:
        continue
    for se in s.senses():
        c = se.counts()
        ili_count[il] += sum(c) if c else 0
    for lem in s.lemmas():
        ili_lemmas[il].add(lem.lower())

freq_en = staged("frequency", "en")
zipf_en = dict(zip(freq_en["word"].str.lower(), freq_en["zipf"], strict=False))


def concept_vs_freq(zipf_map: dict[str, float], lemmas: dict[str, set[str]]):
    xs, ys = [], []
    for il, cnt in ili_count.items():
        zs = [zipf_map[lem] for lem in lemmas[il] if lem in zipf_map]
        if cnt > 0 and zs:
            xs.append(np.log1p(cnt))
            ys.append(max(zs))
    return np.array(xs), np.array(ys)


x, y = concept_vs_freq(zipf_en, ili_lemmas)
print(f"en: {len(x)} concepts; pearson(log SemCor concept count, max en zipf) ="
      f" {np.corrcoef(x, y)[0, 1]:.3f}")
# eyeball head + tail of the concept-commonness ranking
ranked = sorted(ili_count.items(), key=lambda kv: kv[1], reverse=True)
print("top:", [sorted(ili_lemmas[il])[:1] for il, _ in ranked[:8]])

## Cross-language propagation (en signal -> es / it freq)

In [ ]:
# Cross-language propagation: does the *English* concept-commonness signal predict
# the *other* language's lemma frequency? (same ILI -> that language's lemmas)
for lang in ["es", "it"]:
    freq = staged("frequency", lang)
    zipf_map = dict(zip(freq["word"].str.lower(), freq["zipf"], strict=False))
    xs, ys = [], []
    for s in wordnet(lang).synsets():
        il = ili_of(s)
        if not il or ili_count.get(il, 0) <= 0:
            continue
        zs = [zipf_map[lem.lower()] for lem in s.lemmas() if lem.lower() in zipf_map]
        if zs:
            xs.append(np.log1p(ili_count[il]))
            ys.append(max(zs))
    xs, ys = np.array(xs), np.array(ys)
    print(f"{lang}: {len(xs)} concepts; "
          f"pearson(en SemCor concept count, {lang} lemma zipf) ="
          f" {np.corrcoef(xs, ys)[0, 1]:.3f}")

## Findings (measured 2026-06-21)

- **SemCor coverage is partial and very skewed.** 17% of en senses carry a
  count (35,483 / 206,978); median count 2, p90 12, max 10,742. Non-en
  lexicons carry zero counts, as expected.
- **Concept commonness is plausible and validated.** Summing counts to the ILI
  and correlating against the independent en frequency list gives pearson
  **0.47** (log SemCor concept count vs max lemma zipf, 22,139 concepts). The
  head of the ranking is exactly the everyday words.
- **The English signal propagates across languages.** The same English
  concept-commonness predicts *es* lemma frequency at **0.34** and *it* at
  **0.49** - the brief's hypothesis (common concept = common word everywhere)
  holds.

**Decision (routes to phase 6):** carry a concept-level commonness signal
derived from SemCor and use it cross-language; refine per language with that
language's token frequency. Use the English sense split as the phase-6 prior
for languages without sense-tagged corpora. This phase only carries `sense.id`
through the build so phase 6 can join the weights; the weighting math is
phase 6.